# Этап 2a — Candidate Generation: ALS baseline

Матричная факторизация `R ≈ U × Vᵀ` через `implicit.als.AlternatingLeastSquares`.
Пайплайн: id-mapping → обучение ALS → эмбеддинги товаров в Qdrant (`als_items`) →
top-100 retrieval для каждого test-пользователя (с cold-start fallback на popularity
baseline из этапа 1) → Recall@100/NDCG@100 в сравнении с popularity baseline.

См. `docs/plans/stage2_candidate_generation_plan.md`.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.id_mapping import build_id_maps
from src.als_model import ALSModel
from src.vector_store import VectorStore
from src.cold_start import retrieve_candidates
from src.metrics import evaluate_recommendations, evaluate_fixed_recommendation

PROCESSED_DIR = "../data/processed"
K = 100
FACTORS = 64

## Загрузка артефактов этапа 1

In [2]:
train = pd.read_parquet(f"{PROCESSED_DIR}/train.parquet")
test = pd.read_parquet(f"{PROCESSED_DIR}/test.parquet")
popularity_ranking = pd.read_parquet(f"{PROCESSED_DIR}/popularity_ranking.parquet")

train_positive = train[train["is_positive"] == 1]
test_positive = test[test["is_positive"] == 1]

print(f"train: {len(train)} ({len(train_positive)} positive), test: {len(test)} ({len(test_positive)} positive)")

train: 800164 (463017 positive), test: 200045 (112264 positive)


## Id-mapping

Raw MovieLens ID не переиндексированы и не сплошные — строим dense-индекс по
всем train-пользователям/товарам (не только positive), чтобы `known_users`
для cold-start fallback совпадал с определением из этапа 1.

In [3]:
user_map, item_map = build_id_maps(train)
known_users = set(train["user_id"].unique())

print(f"users: {len(user_map)}, items: {len(item_map)}")

users: 5400, items: 3662


## Обучение ALS

In [4]:
als = ALSModel(factors=FACTORS, regularization=0.01, iterations=15, random_state=0)
als.fit(train_positive, user_map, item_map)

print(als.user_factors.shape, als.item_factors.shape)

  0%|          | 0/15 [00:00<?, ?it/s]

(5400, 64) (3662, 64)


## Загрузка item-эмбеддингов в Qdrant

Коллекция `als_items` — отдельная от будущей `two_tower_items`, чтобы можно
было сравнивать оба метода retrieval независимо (этап 4).

In [5]:
store = VectorStore()
store.create_collection("als_items", dim=FACTORS)

item_ids_raw = item_map.to_raw(range(len(item_map)))
store.upsert_items("als_items", item_ids_raw, als.item_factors)

print(f"upserted {len(item_ids_raw)} item vectors into 'als_items'")

upserted 3662 item vectors into 'als_items'


## Retrieval + cold-start fallback

Для cold-start пользователей (нет в train) — popularity baseline из этапа 1,
иначе — ALS-скор через `.recommend()` (implicit сам исключает уже
провзаимодействованные товары).

In [6]:
def als_score_fn(user_id, k):
    user_idx = user_map.raw_to_idx[user_id]
    item_idx, _scores = als.recommend_for_user(user_idx, k=k)
    return item_map.to_raw(item_idx)


test_users = test_positive["user_id"].unique()
recommended_by_user = {
    user_id: retrieve_candidates(user_id, als_score_fn, popularity_ranking, known_users, k=K)
    for user_id in test_users
}

print(f"построены рекомендации для {len(recommended_by_user)} test-пользователей")

построены рекомендации для 1762 test-пользователей


### Sanity-check cold-start ветки

In [7]:
fake_user_id = -1
assert fake_user_id not in known_users
fallback = retrieve_candidates(fake_user_id, als_score_fn, popularity_ranking, known_users, k=K)
assert fallback == popularity_ranking.head(K).index.tolist()
print("cold-start fallback OK:", fallback[:5])

cold-start fallback OK: [2858, 260, 1196, 2028, 1198]


## Метрики: ALS+cold-start retrieval vs popularity baseline

In [8]:
als_metrics = evaluate_recommendations(recommended_by_user, test_positive, K)
popularity_metrics = evaluate_fixed_recommendation(popularity_ranking.head(K).index.tolist(), test_positive, K)

print(f"ALS (+ cold-start fallback) @K={K}: Recall@K={als_metrics['recall@k']:.4f}, NDCG@K={als_metrics['ndcg@k']:.4f} (n_users={als_metrics['n_users']})")
print(f"Popularity baseline    @K={K}: Recall@K={popularity_metrics['recall@k']:.4f}, NDCG@K={popularity_metrics['ndcg@k']:.4f} (n_users={popularity_metrics['n_users']})")

ALS (+ cold-start fallback) @K=100: Recall@K=0.2810, NDCG@K=0.2563 (n_users=1762)


Popularity baseline    @K=100: Recall@K=0.2438, NDCG@K=0.2255 (n_users=1762)
